# Notebook 01: GIS Data Preprocessing & Input Preparation

This notebook handles the initial preprocessing of spatial layers for the Jim Corbett National Park Habitat Suitability Index (HSI) model. We will load the study area boundary (AOI), process Sentinel-2 multispectral bands, calculate the Normalized Difference Vegetation Index (NDVI), and reproject the digital elevation model (DEM) to compute slope.

## Objectives:
1. Read and visualize the Jim Corbett Administrative Boundary (AOI).
2. Clip Sentinel-2 bands (Red, NIR, Blue, Green) to the study area.
3. Calculate the Normalized Difference Vegetation Index (NDVI).
4. Reproject the Digital Elevation Model (DEM) and compute the Slope.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio

# Configure project paths (allowing imports from src/)
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import clean_raster, calculate_ndvi, reproject_raster, calculate_slope

print('Environment set up successfully!')

## 1. Load and Visualize the Study Area (AOI)

We read the Jim Corbett National Park boundary using GeoPandas.

In [ ]:
aoi_path = PROJECT_ROOT / 'data' / 'raw' / 'AOI' / 'Corbett_AOI.shp'
aoi = gpd.read_file(aoi_path)
print(f'Coordinate Reference System (CRS): {aoi.crs}')
aoi.plot(color='lightgreen', edgecolor='black', alpha=0.5)
plt.title('Jim Corbett National Park AOI Boundary')
plt.show()

## 2. Locate Sentinel-2 Multispectral Data

We locate the raw Sentinel-2 bands. If they are not in the repository `data/raw/Sentinel/`, we fall back to the local workspace folder.

In [ ]:
# Locate JP2 band files
sentinel_dir = PROJECT_ROOT / 'data' / 'raw' / 'Sentinel'
fallback_dir = Path(r'C:\Users\DELL\Desktop\gis project\Habitat_Suitability_Corbett\Sentinel\S2A_MSIL2A_20241126T052141_N0511_R062_T44RKT_20241126T083053.SAFE')

jp2_files = list(sentinel_dir.rglob('*.jp2'))
if not jp2_files and fallback_dir.exists():
    print('Using local workspace fallback directory...')
    sentinel_dir = fallback_dir
    jp2_files = list(sentinel_dir.rglob('*.jp2'))

band_files = {}
for file in jp2_files:
    if '_B02_10m' in file.name: band_files['B02'] = file
    elif '_B03_10m' in file.name: band_files['B03'] = file
    elif '_B04_10m' in file.name: band_files['B04'] = file
    elif '_B08_10m' in file.name: band_files['B08'] = file

print('Found bands:')
for b, p in band_files.items():
    print(f' - {b}: {p.name}')

## 3. Clip Sentinel-2 Bands to AOI

We clip the bands using our modular `clean_raster` function, which handles reprojecting the AOI to the raster CRS.

In [ ]:
clean_dir = PROJECT_ROOT / 'data' / 'processed' / 'Cleaned'
clean_dir.mkdir(parents=True, exist_ok=True)

clipped_red = clean_dir / 'B04.tif'
clipped_nir = clean_dir / 'B08.tif'
clipped_blue = clean_dir / 'B02.tif'
clipped_green = clean_dir / 'B03.tif'

clean_raster(band_files['B04'], clipped_red, aoi, nodata_value=0)
clean_raster(band_files['B08'], clipped_nir, aoi, nodata_value=0)
clean_raster(band_files['B02'], clipped_blue, aoi, nodata_value=0)
clean_raster(band_files['B03'], clipped_green, aoi, nodata_value=0)

print('Sentinel bands clipped and saved to data/processed/Cleaned!')

## 4. Calculate Normalized Difference Vegetation Index (NDVI)

NDVI evaluates canopy density and biomass: 

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

In [ ]:
ndvi_clean = clean_dir / 'NDVI_Clean.tif'
calculate_ndvi(clipped_red, clipped_nir, ndvi_clean)

# Visualize NDVI
with rasterio.open(ndvi_clean) as src:
    ndvi_arr = src.read(1)
    
plt.figure(figsize=(8,8))
plt.imshow(ndvi_arr, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
plt.colorbar(label='NDVI Value')
plt.title('Processed NDVI Layer')
plt.axis('off')
plt.show()

## 5. Reproject DEM and Calculate Slope

The raw DEM resolution is 30m. We reproject it to 10m to match the Sentinel grids, clip to the AOI, and calculate the Slope.

In [ ]:
raw_dem = PROJECT_ROOT / 'data' / 'raw' / 'DEM' / 'DEM_30m.tif'
dem_reprojected = clean_dir / 'DEM_Reprojected.tif'
dem_clean = clean_dir / 'DEM_Clean.tif'
slope_clean = clean_dir / 'Slope_Clean.tif'

# 1. Reproject
reproject_raster(raw_dem, clipped_red, dem_reprojected, is_discrete=False)

# 2. Clip
clean_raster(dem_reprojected, dem_clean, aoi, nodata_value=-9999)

# 3. Calculate Slope
calculate_slope(dem_clean, slope_clean)

# Remove temporary reprojected file
if dem_reprojected.exists():
    os.remove(dem_reprojected)

# Visualize Slope
with rasterio.open(slope_clean) as src:
    slope_arr = src.read(1)
    # Set nodata to NaN for plot
    slope_arr[slope_arr == src.nodata] = np.nan
    
plt.figure(figsize=(8,8))
plt.imshow(slope_arr, cmap='YlOrBr', vmin=0, vmax=45)
plt.colorbar(label='Slope (degrees)')
plt.title('Processed Slope Layer')
plt.axis('off')
plt.show()